# Notebook 06: TF-IDF Ranked Retrieval

## KLEAN Chatbot — Restaurant Information Retrieval & Recommendation

### Overview
The **Vector Space Model with TF-IDF** (Term Frequency - Inverse Document Frequency) ranks documents by their relevance to a user query. Documents and queries are mapped to multidimensional vectors, and relevance is measured using **Cosine Similarity**.

**Mathematical Formulas:**
$$\text{TF}(t, d) = 1 + \ln(\text{count}(t, d)) \quad \text{if count} > 0$$
$$\text{IDF}(t) = \ln\left(\frac{N + 1}{\text{DF}(t) + 1}\right) + 1$$
$$\text{Cosine Similarity}(\mathbf{q}, \mathbf{d}) = \frac{\mathbf{q} \cdot \mathbf{d}}{\|\mathbf{q}\|_2 \|\mathbf{d}\|_2} = \sum_{t \in q \cap d} \hat{q}_t \cdot \hat{d}_t$$

**Core Objectives:**
1. Instantiate our custom `TFIDFRetriever` from `src.retrieval.tfidf`.
2. Inspect query vectorization and IDF weights for rare vs common terms.
3. Perform ranked retrieval on diverse queries across the 698-restaurant dataset.
4. Compare custom ranking outputs against Scikit-learn's `TfidfVectorizer` to validate correctness.

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.utils.data_loader import DataLoader
from src.utils.document_converter import DocumentConverter
from src.preprocessing.text_preprocessing import TextPreprocessor
from src.indexing.inverted_index import InvertedIndex
from src.retrieval.tfidf import TFIDFRetriever

print(f"Project root resolved: {PROJECT_ROOT}")

## 1. Build Inverted Index & Initialize TFIDFRetriever

In [ ]:
loader = DataLoader(PROJECT_ROOT / "dataset" / "restaurants.json")
records = loader.load()

converter = DocumentConverter()
docs = converter.convert_all(records)
corpus = converter.get_corpus()

preprocessor = TextPreprocessor()
tokenized_corpus = preprocessor.preprocess_documents(corpus)

index = InvertedIndex()
index.build(tokenized_corpus)

tfidf_retriever = TFIDFRetriever(index, preprocessor)
print(f"TF-IDF model initialized across {index.num_documents} documents.")

## 2. Inspect IDF Weights (Rare vs. Frequent Terms)

Rare words like `'ramen'` or `'pizza'` receive higher IDF weights than common terms like `'restaurant'`.

In [ ]:
terms_to_check = ["restaurant", "cambodian", "soup", "noodles", "ramen", "pizza", "amok"]
idf_table = []
for term in terms_to_check:
    df_val = index.get_document_frequency(term)
    idf_val = tfidf_retriever.get_idf(term)
    idf_table.append({"Term": term, "Document Frequency (DF)": df_val, "IDF Weight": round(idf_val, 4)})

print(pd.DataFrame(idf_table).to_string(index=False))

## 3. Query Vectorization & Cosine Similarity Ranking

Search for *"traditional cambodian noodle soup"*.

In [ ]:
query = "traditional cambodian noodle soup"
results = tfidf_retriever.search(query, top_k=5)

print(f"Top 5 TF-IDF Results for: '{query}'\n")
for r in results:
    rec = converter.get_record(r["doc_id"])
    print(f"Rank {r['rank']} [Score: {r['score']:.4f}] -> [{r['doc_id']}] {rec['name']}")
    print(f"   Category: {rec['category']} | Subcategories: {rec['sub_category']}")
    print(f"   Address:  {rec['address']}\n")

## 4. Validation: Custom TF-IDF vs. Scikit-Learn

Compare our from-scratch implementation with `sklearn.feature_extraction.text.TfidfVectorizer`.

In [ ]:
test_query = "italian pizza pasta"
comparison = tfidf_retriever.compare_with_sklearn(corpus, query=test_query, top_k=5)

print(f"Query: '{test_query}'")
print(f"Top-5 Overlap: {comparison['top_k_overlap_count']} / 5 ({comparison['top_k_overlap_percentage']}%)")

print("\n--- Custom TF-IDF Top 5 ---")
for r in comparison["custom_rankings"]:
    name = converter.get_record(r["doc_id"])["name"]
    print(f"Rank {r['rank']} [Score: {r['score']:.4f}] -> {name}")

print("\n--- Scikit-Learn TF-IDF Top 5 ---")
for r in comparison["sklearn_rankings"]:
    name = converter.get_record(r["doc_id"])["name"]
    print(f"Rank {r['rank']} [Score: {r['score']:.4f}] -> {name}")

## 5. Observations & Takeaways
- TF-IDF successfully transitions from unranked binary matching to continuous relevance scoring.
- Rare terms contribute significantly more to the score due to smooth IDF weighting.
- Custom implementation matches Scikit-learn's rankings closely.
- **Limitation**: TF-IDF does not have term frequency saturation (a document mentioning 'soup' 20 times can score excessively high), and does not account for document length normalization as robustly as BM25.
- **Next Step**: In `07_bm25.ipynb`, we implement Okapi BM25 to address these limitations!